# Complexity-Escalation phase comparison

Overlays every phase's frozen artifact (`data/phase_comparison/<phase>.npz`, written by
`experiments/phase_comparison.py`) on each chart. Contract + methodology: **ADR-0015**;
diversity terms: `experiments/CONTEXT.md`.

All phases share a fixed env-step budget (3000 epochs of the Phase-0 config) and 5 seeds; every
quantity here is a measured *output*. Add a phase by dropping its `.npz` in `data/phase_comparison/`
and appending a `### Phase N` interpretation cell at the bottom.

In [ ]:
import glob, os
import numpy as np
import matplotlib.pyplot as plt

DATA = os.path.join('..', 'data', 'phase_comparison')
ARTIFACTS = {}
for f in sorted(glob.glob(os.path.join(DATA, '*.npz'))):
    d = np.load(f, allow_pickle=True)
    ARTIFACTS[str(d['phase'])] = d
PHASES = list(ARTIFACTS)
print('phases:', PHASES)

def ms(phase, key):
    """(mean, std) for a per-seed scalar; nan if absent."""
    d = ARTIFACTS[phase]
    m = d[f'mean_{key}'] if f'mean_{key}' in d else np.nan
    s = d[f'std_{key}'] if f'std_{key}' in d else 0.0
    return float(m), float(s)

def bar(keys, title, ylabel, source='seed'):
    """Grouped bars over phases for one or more per-phase scalars (mean +- std)."""
    x = np.arange(len(PHASES)); w = 0.8 / max(1, len(keys))
    fig, ax = plt.subplots(figsize=(1.6 * len(PHASES) + 3, 4))
    for j, key in enumerate(keys):
        if source == 'between':
            ys = [float(ARTIFACTS[p][f'between_{key}']) if f'between_{key}' in ARTIFACTS[p] else np.nan for p in PHASES]
            es = None
        else:
            mv = [ms(p, key) for p in PHASES]; ys = [a for a, _ in mv]; es = [b for _, b in mv]
        ax.bar(x + j * w, ys, w, yerr=es, capsize=3, label=key)
    ax.set_xticks(x + w * (len(keys) - 1) / 2); ax.set_xticklabels(PHASES, rotation=20, ha='right')
    ax.set_title(title); ax.set_ylabel(ylabel); ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

## Performance — eval-time control return on the converged generator

In [ ]:
bar(['perf_top', 'perf_distavg', 'perf_topk'], 'Performance (det-mu return)', 'raw episode return')

## Runtime & convergence

In [ ]:
bar(['steps_per_sec'], 'Throughput', 'env-steps / sec')
bar(['peak_mem_mib'], 'Peak GPU memory', 'MiB')
bar(['conv_quality', 'conv_morph'], 'Convergence', 'env-steps to converge')

## Diversity (within-run and between-seed)

In [ ]:
bar(['div_comp', 'div_struct', 'div_nmodes'], 'Within-run diversity', 'value')
bar(['div_comp', 'div_struct', 'div_nmodes', 'mode_overlap'], 'Between-seed diversity', 'value', source='between')

## Training curves (overlaid, per-seed)

In [ ]:
def overlay_curve(tag, ylabel):
    key = tag.replace('/', '_')
    fig, ax = plt.subplots(figsize=(8, 4))
    for p in PHASES:
        d = ARTIFACTS[p]; drawn = False
        for sk in [k for k in d.files if k.startswith('curve__') and k.endswith(f'{key}__step')]:
            st = d[sk]; vl = d[sk[:-6] + '__val']
            ax.plot(st, vl, alpha=0.5, label=p if not drawn else None,
                    color=f'C{PHASES.index(p)}'); drawn = True
    ax.set_title(tag); ax.set_xlabel('env-steps'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

overlay_curve('quality/R_mean', 'body quality R (scaled)')
overlay_curve('build/limbcount', 'mean limb count')

## Per-phase interpretation

One section per phase: what the phase changed, what the charts above show, and what it means.

### Phase 0 — Baseline (presence-only ant codesign)

_Fill in after the first 5-seed run:_ headline eval-return (top / dist-avg / top-K), baseline
throughput + peak-mem, quality vs morphology convergence steps, and the diversity floor
(within-run `N_modes`, between-seed mode-overlap) that later phases are measured against.